# Lesson 01 — Naive RAG, end to end

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/safar-byte/rag-from-scratch-to-production/blob/main/lessons/01-naive-rag/colab.ipynb)

*The whole pipeline at its simplest — built first so there is something concrete to criticise.*

**What you will learn**

- Build a complete RAG pipeline **from scratch** in ~40 lines of numpy: load, chunk, embed, search, prompt, generate
- Do the same with `ragkit` and read every stage it exposes
- Why cosine (not L2), why similarity (not distance), and why the prompt says "answer only from the context"
- How to attack a RAG system — identifiers, refusals, top-k — and why a small corpus makes every attack look like it failed

| | |
|---|---|
| ⏱️ Time | 45–60 min |
| 🖥️ Runtime | CPU is fine (T4 GPU makes generation ~10× faster) |
| 🤖 Needs the generator (Ollama) | yes — for some cells |

**Before this:** [00-setup](https://colab.research.google.com/github/safar-byte/rag-from-scratch-to-production/blob/main/lessons/00-setup/colab.ipynb)

> **How to use this notebook.** Run the cells top to bottom (`Shift+Enter`). Every
> time you see **🔮 Predict**, stop and write down what you expect *before* running
> the next cell — the gap between prediction and result is where the learning is.
> Answers to the check-your-understanding questions are hidden behind ▶ toggles.

## 0. Setup

This cell clones the course repo into the Colab runtime, installs the
dependencies, and (if `USE_LLM` is ticked) installs and starts
[Ollama](https://ollama.com) with the small `qwen2.5:1.5b` generator. It is safe
to re-run. Everything in the repo's `ragkit/` library becomes importable.

If an import fails after the install, use **Runtime → Restart session** and run
this cell again.

In [ ]:
# @title ⚙️ Setup — run me first
USE_LLM = True  # @param {type:"boolean"}
BRANCH = "main"  # @param {type:"string"}

import subprocess
import sys
from pathlib import Path


def _find_repo():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "ragkit").is_dir() and (p / "colab" / "colab_setup.py").exists():
            return p
    return None


REPO = _find_repo()
if REPO is None:
    REPO = Path("/content/rag-from-scratch-to-production")
    if not REPO.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                BRANCH,
                "https://github.com/safar-byte/rag-from-scratch-to-production.git",
                str(REPO),
            ],
            check=True,
        )
sys.path.insert(0, str(REPO / "colab"))
from colab_setup import compare_runs, eval_frames, results_frame, setup  # noqa: F401

LLM_READY = setup(llm=USE_LLM)

## 1. The five steps

| Step | What happens | Where in `ragkit` |
|---|---|---|
| 1. **Load** | read documents off disk | `ingest/loader.py` |
| 2. **Chunk** | split them into retrievable units | `ingest/chunker.py` |
| 3. **Embed** | turn each chunk into a vector | `providers/` |
| 4. **Store** | index the vectors for nearest-neighbour search | `store/chroma.py` |
| 5. **Retrieve & generate** | embed the question, take the top *k*, put them in a prompt | `retrieve/dense.py`, `generate/` |

We will build each step by hand first, so you see there is no magic, and then switch
to the library.

## 2. RAG from scratch

### Step 1 — Load
A document is just text plus an id. We keep the file name as the id: a citation that
says `04-reranking.md` tells you something; one that says `a3f9c1` does not.

In [ ]:
from pathlib import Path

docs = {p.name: p.read_text() for p in sorted(Path("data").glob("*.md")) if p.name != "README.md"}
print(len(docs), "documents,", sum(map(len, docs.values())), "characters")

### Step 2 — Chunk
Whole documents are too big to embed well (one vector can't represent twelve topics)
and too big to stuff into a prompt. The crudest useful chunker: split on blank lines
(paragraphs) and merge small paragraphs until a size limit. Lesson 02 does this
properly.

In [ ]:
def naive_chunks(text: str, max_chars: int = 600) -> list[str]:
    chunks, buffer = [], ""
    for para in text.split("\n\n"):
        if buffer and len(buffer) + len(para) > max_chars:
            chunks.append(buffer)
            buffer = ""
        buffer = f"{buffer}\n\n{para}" if buffer else para
    if buffer:
        chunks.append(buffer)
    return chunks


chunks = [(doc_id, c) for doc_id, text in docs.items() for c in naive_chunks(text)]
print(len(chunks), "chunks")
print("---- example chunk ----")
print(chunks[5][1])

### Step 3 — Embed
One vector per chunk, through the `Embedder` protocol from lesson 00. The vectors are
normalised to length 1, so a dot product *is* the cosine similarity.

In [ ]:
import numpy as np

from ragkit.providers import get_embedder

embedder = get_embedder()
matrix = np.array(embedder.embed_documents([c for _, c in chunks]))
print("embedding matrix:", matrix.shape, "(chunks × dimensions)")

### Step 4 — Store, and Step 5a — Search
For 100-odd vectors, "the store" can be a numpy array and "search" can be one matrix
multiplication followed by a sort. That is **exact nearest-neighbour search**. Real
vector databases use an *approximate* index (HNSW, see `data/08-vector-indexes.md`)
because sorting millions of scores per query is too slow — but the result is the same
idea.

In [ ]:
def search(question: str, k: int = 3):
    q = np.array(embedder.embed_query(question))
    scores = matrix @ q  # cosine similarity with every chunk
    top = np.argsort(-scores)[:k]  # highest first
    return [(float(scores[i]), *chunks[i]) for i in top]


for score, doc_id, text in search("What is reciprocal rank fusion?"):
    print(f"{score:.3f}  {doc_id:28s} {text[:80]!r}")

### Step 5b — Prompt and generate
Number the passages so the model can cite them, put them **before** the question
(lesson 12 explains why that order matters for caching), and tell the model to say
when the context is not enough.

In [ ]:
SYSTEM = (
    "Answer using only the numbered context passages. Cite them like [1]. "
    "If the context does not contain the answer, say so."
)


def scratch_prompt(question: str, hits) -> str:
    context = "\n\n".join(
        f"[{i}] ({doc_id})\n{text}" for i, (_, doc_id, text) in enumerate(hits, 1)
    )
    return f"Context passages:\n\n{context}\n\nQuestion: {question}\n\nAnswer:"


question = "What is reciprocal rank fusion and why does it use ranks instead of scores?"
prompt = scratch_prompt(question, search(question))
print(prompt[:1200], "…")

In [ ]:
if LLM_READY:
    from ragkit.providers import get_generator

    generator = get_generator()
    answer, usage = generator.generate(system=SYSTEM, prompt=prompt, max_tokens=300)
    print(answer)
    print(f"\n[{usage.input_tokens} prompt tokens, {usage.output_tokens} answer tokens]")
else:
    print("Generator not available — retrieval above is the half worth watching anyway.")

🎉 **That is a working RAG system.** Everything else in this course is a response to a
specific way this version fails.

## 3. The same pipeline with `ragkit`

`RagPipeline` wires the same five steps together, with the details that a from-scratch
version gets wrong: real chunking, a persistent vector store (Chroma), a keyword index
built from the same chunks (lesson 04), provenance on every chunk, and a trace of what
happened.

In [ ]:
from ragkit.pipeline import RagPipeline

pipeline = RagPipeline()  # strategy="dense" — this lesson's pipeline
pipeline._embedder = embedder  # reuse the model we already loaded
stats = pipeline.ingest()  # load → chunk → embed → store (resets the index)
stats

`ingest()` resets the index every time. That is a deliberate choice recorded in the
repo's decision log: partially re-indexing a corpus whose chunk boundaries moved leaves
orphaned chunks behind, and *a stale index is far harder to debug than a slow rebuild*.

Now retrieve. Each result is a `Scored`: the chunk, a score, and a dict of per-stage
scores (only `dense` so far).

In [ ]:
question = "What is reciprocal rank fusion?"
results = pipeline.retrieve(question, top_k=3)
results_frame(results, chars=120)

In [ ]:
top = results[0]
print("chunk_id :", top.chunk.chunk_id)
print("doc_id   :", top.chunk.doc_id, "| title:", top.chunk.title)
print("offsets  :", top.chunk.start_char, "→", top.chunk.end_char, "in the source document")
print("scores   :", top.scores)

The character offsets look like fussy bookkeeping. They are what make citations
checkable and what make lesson 08 (parent-document retrieval) a simple slice instead
of a second index.

### The prompt `ragkit` builds

In [ ]:
from ragkit.generate import SYSTEM_PROMPT, build_prompt

print("SYSTEM PROMPT\n" + "=" * 60 + "\n" + SYSTEM_PROMPT)
print("USER PROMPT\n" + "=" * 60 + "\n" + build_prompt(question, results))

### Ask
`pipeline.ask()` = retrieve → build prompt → generate → parse `[n]` citations. The
returned `Answer` carries the text, the citations, the contexts it was grounded in,
token usage, and a timing trace.

In [ ]:
if LLM_READY:
    answer = pipeline.ask(question, top_k=3)
    print(answer.text)
    print("\ncited:", [c.doc_id for c in answer.citations])
    print("trace:", answer.trace)

## 4. Details that are not incidental

### Cosine, not L2
Chroma's collection is created with `{"hnsw:space": "cosine"}`. Euclidean (L2) distance
is affected by vector *length*, and length correlates with things like document length
that should not affect relevance. Cosine only looks at direction.

### Similarity, not distance
Chroma returns cosine **distance** (`1 − similarity`, lower is better). `ChromaStore`
flips it so that *higher is better everywhere* in the codebase. Mix the two conventions
and you get a retriever that confidently returns the **least** relevant chunks — and the
code reads fine. Let's see the raw values Chroma returns:

In [ ]:
raw = pipeline._store._collection.query(
    query_embeddings=[embedder.embed_query(question)], n_results=3, include=["distances"]
)
for chunk_id, distance in zip(raw["ids"][0], raw["distances"][0], strict=True):
    print(f"{chunk_id:36s} distance={distance:.4f}  →  similarity={1 - distance:.4f}")

### "Answer only from the context" is necessary but not sufficient
Without the instruction, the model happily answers from its training data — the exact
failure RAG exists to prevent. With it, the model *still sometimes does*. Let's probe it
with a question the corpus cannot answer.

🔮 **Predict:** the corpus is about retrieval systems. Asked "What is the capital of
France?", will the model refuse, or say "Paris"?

In [ ]:
probe = "What is the capital of France?"
results_frame(pipeline.retrieve(probe, top_k=3))

In [ ]:
if LLM_READY:
    for attempt in range(3):
        print(f"attempt {attempt + 1}:", pipeline.ask(probe, top_k=3).text, "\n")

Notice two things:

1. **Retrieval always returns *something*.** There is no "no results" in vector search —
   the top 3 are simply the least-unrelated chunks, with lower scores. Nothing tells the
   model they are irrelevant except the scores, which it never sees.
2. Temperature is 0, so all three attempts are identical. That is deliberate: lesson
   03's measurements are meaningless if the same input gives different outputs.

Whether it refused or not, you have one data point. Lesson 03 replaces this with a
measured refusal rate over ten unanswerable questions — and finds the real number is
much worse than one probe suggests.

## 5. Try to break it

### Attack 1 — identifiers
The textbook weakness of dense retrieval: an identifier like `TX-4491` carries almost no
meaning for an embedding model. So this should fail.

🔮 **Predict** the rank of the chunk that contains `TX-4491` before running the cell.

In [ ]:
import pandas as pd


def rank_of(text_fragment: str, question: str, depth: int = 20):
    for item in pipeline.retrieve(question, top_k=depth):
        if text_fragment in item.chunk.text:
            return item.rank
    return None


attacks = [
    ("TX-4491", "What is error code TX-4491?"),
    ("75", "What does exit status 75 mean?"),
    ("FUSION_CONSTANT", "What is FUSION_CONSTANT set to?"),
    ("8788", "Which port does the query service use?"),
]
pd.DataFrame(
    [
        {"query": q, "looking for": frag, "rank of chunk containing it": rank_of(frag, q)}
        for frag, q in attacks
    ]
)

They all come back at or near rank 1. Two things are going on, and both are worth more
than a successful demo would have been:

**The corpus is too small.** About 110 chunks. Asking for the top 5 means asking for ~5%
of everything, and at that ratio almost any retrieval method looks excellent. This is
the **ceiling effect**, and it is the single most common way RAG benchmarks lie. A
technique that shows no improvement on a corpus this size has not been shown to be
useless — it has been shown to be *untestable here*.

**The prose around the identifier gives it away.** The chunk with `TX-4491` sits in a
paragraph *about* error codes, so the words "error code" in the question match
semantically. A rigged demonstration.

Let's make the attack fairer: strip the helpful words out of the question.

In [ ]:
for q in ["TX-4491", "status 75", "FUSION_CONSTANT", "8788"]:
    hits = pipeline.retrieve(q, top_k=3)
    print(f"{q!r:20s} →", [f"{h.chunk.doc_id} ({h.score:.2f})" for h in hits])

With bare identifiers the similarity scores drop and the right document is no longer a
sure thing. Lesson 04 adds BM25 keyword search for exactly this case.

### Attack 2 — top-k
How many passages should go in the prompt? More passages = more chance the answer is
there, but also more distraction and more tokens. Look at what `top_k = 10` actually
puts in front of the model for a narrow question:

In [ ]:
results_frame(pipeline.retrieve("What is BM25?", top_k=10), scores=False)

Count how many of those ten are actually *about* BM25. The rest is noise the model must
ignore. The fraction that is relevant is **precision@k**; the fraction of relevant
passages you found is **recall@k** — both defined precisely in lesson 03. Reranking
(lesson 05) exists to put the relevant ones on top.

You cannot yet say rigorously which `k` is best — you are eyeballing it. Sit with how
unsatisfying that is. It is the entire argument for lesson 03.

## 6. What is wrong with this pipeline

| Problem | Fixed in |
|---|---|
| Fails on identifiers, error codes, rare terms | 04 — hybrid search |
| Top-k by cosine is a weak ranking | 05 — reranking |
| The user's question is often a poor search query | 06 — query transformation |
| Chunks that are meaningless out of context | 07 — contextual retrieval |
| Precise retrieval and full context are in tension | 08 — parent-document |
| No way to tell whether retrieval even worked | 09–10 — grading and correction |
| Cannot answer questions spanning two documents | 10–11 — multi-hop, GraphRAG |
| No caching, no citations, no cost visibility | 12 — production |
| **No idea whether any change makes it better** | **03 — evaluation** |

The last row is the one that matters. Everything above it is a guess until you can
measure.

## 7. Check your understanding

**1. Why does the system prompt tell the model to say when the context is insufficient, and why is that not enough on its own?**

<details><summary>▶ Show answer</summary>

Without it, the model answers from parametric (training) knowledge, which defeats
the purpose of RAG and produces unverifiable answers. It is not enough because an
instruction is a request, not a control: small models in particular still answer
confidently from irrelevant context. You need to *measure* refusals (lesson 03) and
add real controls (a score floor, a grader — lessons 10 and 12).

</details>

**2. A teammate reports: "dense retrieval found every identifier at rank 1, so we don't need keyword search." What do you say?**

<details><summary>▶ Show answer</summary>

On a ~110-chunk corpus top-5 is ~5% of everything, so almost any method looks
perfect (the ceiling effect), and the test questions contained helpful words around
the identifier. Nothing has been shown except that this corpus cannot distinguish
the methods. Test with bare identifiers, on a bigger corpus, with a proper metric.

</details>

**3. Chroma returns a *distance*. What goes wrong if you sort by it descending, as if it were a similarity?**

<details><summary>▶ Show answer</summary>

You return the *least* relevant chunks first. Nothing errors and the code looks
normal — the answers just get mysteriously bad. That is why `ragkit` converts to
similarity at the store boundary and uses "higher is better" everywhere else.

</details>

## ✍️ Exercises

1. **Sweep top-k.** Run `pipeline.ask("How does chunk size affect retrieval?", top_k=k)`
   for k = 1, 3, 5, 10. Where does the answer stop improving and start getting buried?
2. **Probe refusals.** Ask three questions the corpus cannot answer (e.g. "Who won the
   2018 World Cup?"). Record how often the model refuses cleanly, answers anyway, or
   hedges. That ratio is your informal before-measurement for lesson 03.
3. **Break the from-scratch version.** Change `scores = matrix @ q` to
   `scores = -(matrix @ q)` (a "distance"). Re-run `search`. How long would it take you
   to notice in a real system?

In [ ]:
# Exercise 1 — your turn
if LLM_READY:
    for k in (1, 3, 5, 10):
        a = pipeline.ask("How does chunk size affect retrieval?", top_k=k)
        print(f"--- top_k={k} ({a.trace['prompt_chars']} prompt chars)\n{a.text}\n")

## Key takeaways

1. RAG from scratch is ~40 lines: chunk, embed, matrix-multiply, sort, prompt, generate.
2. Keep provenance (doc id, offsets) on every chunk — citations and later lessons depend on it.
3. Cosine similarity, higher-is-better everywhere; mixing distance and similarity fails silently.
4. Vector search always returns *something*; the prompt instruction to refuse is necessary but not sufficient.
5. A small corpus hides failures (the ceiling effect) — be suspicious of easy wins.

## Next

**[Lesson 02 — Chunking](https://colab.research.google.com/github/safar-byte/rag-from-scratch-to-production/blob/main/lessons/02-chunking/colab.ipynb)**